# sfig15 — Modality Importance Radar Chart (Supplementary Fig. S-15)

Polar/radar chart: each axis = |ΔAUROC| when that modality is removed.
One polygon per task. 5 spokes: No BAS, No RESP, No EKG, Cardio only, BAS only.

**Data**: `results/tables/table6_modality.csv`.

In [ ]:
%matplotlib inline
%load_ext autoreload
%autoreload 2

In [ ]:
import sys
from pathlib import Path

# ── Workspace root ────────────────────────────────────────────────────────────
def _find_workspace():
    candidate = Path.cwd().resolve()
    for _ in range(10):
        if (candidate / "final_results").exists():
            return candidate
        if candidate.parent == candidate:
            break
        candidate = candidate.parent
    return Path("/Users/boshra/NSRR-workspace").resolve()

WORKSPACE_ROOT = _find_workspace()
NSRR_TOOLS     = WORKSPACE_ROOT / "NSRR-tools"
PAPER_FIGURES  = NSRR_TOOLS / "results" / "paper_figures"
FINAL_OUT      = PAPER_FIGURES / "final"
FINAL_OUT.mkdir(parents=True, exist_ok=True)

# Main utils (TBME style, data, panels)
_nb_dir = PAPER_FIGURES / "notebooks"
sys.path.insert(0, str(_nb_dir))
# Explore utils (panel functions not yet ported to main panels.py)
_explore_nb_dir = PAPER_FIGURES / "explore" / "notebooks"
sys.path.insert(0, str(_explore_nb_dir))

from utils.style import (
    apply_tbme_style, save_figure, FULL_W, HALF_W,
    MAIN_TASKS, SUPP_TASKS, ALL_TASKS,
    TASK_LABEL, FONT_BASE, FONT_LABEL, CTX_ORDER,
)
from utils.data import set_root, load_analysis, load_heatmap, load_parquets
from utils.data_explore import load_modality_table   # only in explore utils
from utils import panels
from utils import panels_explore as xp   # panel functions not yet in panels.py

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

set_root(WORKSPACE_ROOT)
apply_tbme_style()

_ok = (WORKSPACE_ROOT / "final_results").exists()
print(f"WORKSPACE_ROOT : {WORKSPACE_ROOT}")
print(f"final_results/ : {'✓ found' if _ok else '✗ NOT FOUND — edit _find_workspace() fallback'}")

In [ ]:
# ── Load modality table ───────────────────────────────────────────────────────
mod_df = load_modality_table(NSRR_TOOLS)
print(mod_df.to_string())
print("\nColumns:", mod_df.columns.tolist())

In [ ]:
# Task names in the table vs internal keys — map them
TASKS = MAIN_TASKS

# Map internal task keys to the 'Task' column values in table6_modality.csv
TASK_TABLE_MAP = {
    "sex_binary":                "Sex",
    "apnea_binary":              "Sleep apnea",
    "sleep_efficiency_binary":   "Sleep efficiency",
    "age_class":                 "Age",
    "bmi_binary":                "BMI",
}

fig = plt.figure(figsize=(5.5, 5.5))
ax = fig.add_subplot(111, projection="polar")

handles, labels = xp.modality_radar_panel(ax, mod_df, tasks=TASKS)
ax.legend(handles, [TASK_LABEL.get(t, t) for t in TASKS],
          loc="upper left", bbox_to_anchor=(1.15, 1.1),
          fontsize=7, frameon=False)
ax.set_title("Modality importance: |ΔAUROC| when modality removed",
             fontsize=8, pad=15)
fig.tight_layout()
plt.show()

In [ ]:
# ── Run when figure looks good ────────────────────────────────────────────────
save_figure(fig, FINAL_OUT, "sfig15_modality_radar")
import shutil
shutil.copy(FINAL_OUT / "sfig15_modality_radar.pdf",
            WORKSPACE_ROOT / "TBME_submission" / "sfig15_modality_radar.pdf")
print("Saved + copied → TBME_submission/sfig15_modality_radar.pdf")